<div style="border-top: 1px solid rgba(128, 128, 128, 0.45); padding-top: 10px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65;">Walkthrough &middot; about 3 min</span></div>

## <span style="font-family: Newsreader, Georgia, serif; font-size: 34px; font-weight: 400;">5. Results & Visualization</span>

<div style="font-size: 19px; line-height: 1.5; max-width: 44em;">How both runs score on their test images beside the paper's result, then what the concept explanations reveal about single predictions, right and wrong.</div>

### 5.1 Final test accuracy against the paper

Each run was tested once, with the checkpoint that had the lowest validation cross-entropy. The paper's CUB section describes no validation split, so the like-for-like comparison with its Table 1 is the repository-split run, which keeps the repository's own split and settings.

In [ ]:
# Headline: test accuracy of both runs' selected checkpoints, beside the paper's reported CUB result.
PAPER_CT_ACCURACY = (88.0, 0.4)  # cited constant: Table 1 of 2647_attention_based_interpretabili.pdf, "CT", test accuracy (%)
# (Table 1 also reports 76.9 ± 3 for the same model trained without concepts, "CT [w/o]".)

from html import escape

paper_mean, paper_spread = PAPER_CT_ACCURACY
test_accuracy = {job: 100 * RUNS[job]["snapshot"]["test_metrics"]["test_acc"] for job in (REPO_RUN, DISJOINT_RUN)}
tiles = [{"label": "Paper · Table 1", "value": f"{paper_mean:.1f} ± {paper_spread:.1f}",
          "detail": "Concept Transformer on CUB, test accuracy (%)", "color": "rule"}]
for job in (REPO_RUN, DISJOINT_RUN):
    snapshot = RUNS[job]["snapshot"]
    tiles.append({
        "label": f"{RUN_LABEL[job]} · test accuracy",
        "value": f"{test_accuracy[job]:.2f}%",
        "detail": (f"{test_accuracy[job] - paper_mean:+.2f} points vs paper · epoch {snapshot['best_validation_epoch']}"
                   f" · {snapshot['dataset_sizes']['test']:,} test images"),
        "color": RUN_COLOR[job],
    })
stat_row(tiles)

gap = test_accuracy[REPO_RUN] - paper_mean
verdict = (f"Like for like, the repository-split run lands {abs(gap):.2f} percentage points "
           f"{'below' if gap < 0 else 'above'} the paper's {paper_mean:.1f}%, "
           f"{'inside' if abs(gap) <= paper_spread else 'outside'} its reported ±{paper_spread:.1f}, from one training seed.")
display(HTML(
    f'<div style="border-top: 3px solid {PALETTE["accent"]}; background: rgba(172, 84, 63, 0.07); '
    'padding: 12px 18px 14px 18px; margin-top: 18px;">'
    '<div style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65; margin-bottom: 6px;">'
    'Reproduction verdict</div>'
    f'<div style="font-family: Newsreader, Georgia, serif; font-size: 21px; line-height: 1.4;">{escape(verdict)}</div></div>'
))

In [ ]:
# Evidence: each selected checkpoint's validation and test metrics, read from the saved run records.
metrics = {}
for job in (REPO_RUN, DISJOINT_RUN):
    snapshot = RUNS[job]["snapshot"]
    test, validation, sizes = snapshot["test_metrics"], snapshot["best_validation_metrics"], snapshot["dataset_sizes"]
    metrics[f"{RUN_LABEL[job]} ({job})"] = {
        "Selected epoch (lowest validation CE)": str(snapshot["best_validation_epoch"]),
        "Validation accuracy at that epoch": f"{100 * validation['val_acc']:.2f}%",
        "Test accuracy": f"{100 * test['test_acc']:.2f}%",
        "Test classification CE": f"{test['test_ce_loss']:.4f}",
        "Test explanation loss": f"{test['test_expl_loss']:.4f}",
        "Test attention-concentration penalty": f"{test['test_l1_loss']:.4f}",
        "Train / validation / test images": f"{sizes['train']:,} / {sizes['validation']:,} / {sizes['test']:,}",
    }
display(pd.DataFrame(metrics))

<div style="border-top: 3px solid #be7c1c; background: rgba(190, 124, 28, 0.08); padding: 10px 18px 12px 18px;"><div style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65; margin-bottom: 4px;">Caveat</div><div style="font-size: 15px; line-height: 1.5;">The gap between our two runs is not a measured effect of removing the validation&ndash;test overlap: they train and test on different images (Section 3), and each used one seed. Table 1 does not say how many runs its &plusmn; spans either, and one run cannot show whether a gap of this size is real.</div></div>

### 5.2 Six test predictions, explained by their concepts

Six images from the disjoint run's test split, scored by its selected checkpoint. A fixed rule chose them, not us: the first three correct and the first three wrong predictions found at evenly spaced test positions. Each panel shows the spatial concept with the strongest attention in that image.

In [ ]:
# All six cached examples at a glance, as rendered by scripts/visualize_test_explanations.py.
show_image(ARTIFACT_DIR / "overview.png", width=900, caption_text=(
    "<b>Figure 5.1</b> Three correct (green titles) and three wrong (red titles) test predictions from the disjoint run, "
    "with predicted probabilities. Each image shows its strongest spatial concept on a fixed 0–1 attention scale over "
    "the 14 × 14 patch grid; cyan rings mark patches where CUB annotates that attribute. Under each image: the "
    "spatial concept shown and the two global concepts with the most attention."))

In [ ]:
# Evidence: the cached examples come from the disjoint run's selected checkpoint, and their contributions rebuild every logit.
explanations = json.loads((ARTIFACT_DIR / "examples.json").read_text())
with np.load(ARTIFACT_DIR / "selected_examples.npz") as saved:
    tensors = dict(saved)
disjoint_job = EVIDENCE["disjoint_training_job"]
assert explanations["job_id"] == DISJOINT_RUN
assert explanations["manifest_sha256"] == disjoint_job["manifest_sha256"]
assert explanations["checkpoint_path"] == disjoint_job["completion_verification"]["checkpoint_path"]
assert explanations["checkpoint_epoch_one_based"] == RUNS[DISJOINT_RUN]["snapshot"]["best_validation_epoch"]
assert explanations["split_counts"] == RUNS[DISJOINT_RUN]["snapshot"]["dataset_sizes"]
examples = explanations["examples"]
assert len(examples) == 6 and sum(sample["correct"] for sample in examples) == 3
rebuilt = tensors["contributions"].sum(axis=1) + tensors["bias"]  # sum over the concepts, plus the classifier biases
np.testing.assert_allclose(rebuilt, tensors["logits"], atol=1e-4, rtol=1e-4)
for index, sample in enumerate(examples):
    assert sample["correct"] == (sample["true_label"] == sample["predicted_label"])
    assert int(tensors["logits"][index].argmax()) == sample["predicted_label"]
    contributions = tensors["contributions"][index]
    margin = (contributions[:, sample["predicted_label"]] - contributions[:, sample["comparison_label"]]).sum()
    np.testing.assert_allclose(margin + sample["bias_margin"], sample["logit_margin"], atol=1e-4)
    assert (ARTIFACT_DIR / sample["figure_png"]).is_file()

display(pd.DataFrame([{
    "Outcome": "Correct" if sample["correct"] else "Wrong", "Image": sample["image_id"],
    "True species": sample["true_class"], "Predicted species": sample["predicted_class"],
    "Predicted probability (%)": 100 * sample["predicted_probability"],
    "True-class probability (%)": 100 * sample["true_probability"],
    "Compared with": sample["comparison_class"], "Logit margin": sample["logit_margin"],
} for sample in examples]).set_index(["Outcome", "Image"]).style.format({
    "Predicted probability (%)": "{:.1f}", "True-class probability (%)": "{:.2f}", "Logit margin": "{:.2f}",
}))
n_concepts, n_classes = tensors["contributions"].shape[1:]
print(f"Verified against the run records: job {explanations['job_id']}, checkpoint from epoch "
      f"{explanations['checkpoint_epoch_one_based']}, {explanations['split_counts']['test']:,}-image test split.")
print(f"{n_concepts} concept contributions plus the classifier biases rebuild all {n_classes} logits of each example; "
      f"largest error recorded at extraction: {explanations['maximum_logit_reconstruction_error']:.1e}.")

### 5.3 Reading one explanation, right and wrong

Each detail figure answers three questions: where the model looks, what it sees in the whole bird, and which concepts decide.

1. **Top row, where it looks:** the three spatial concepts with the highest peak attention, each from a different attribute family. Color is attention per 16 × 16-pixel patch on a fixed 0–1 scale; cyan rings mark patches where CUB annotates that attribute.
2. **Bottom left, what it sees:** the five global concepts with the most attention. Attention is a share, not a probability: the 13 global weights sum to 1, as do the 95 spatial weights at each patch. A bar at 0.3 means 30% of the attention, not a 30% chance that the bird has that color; an asterisk marks an attribute annotated for this image.
3. **Bottom right, what decides:** each concept's signed contribution to the logit gap between the prediction and a comparison class, the runner-up when the prediction is right and the true class when it is wrong. Green pushes toward the prediction, red toward the comparison.

The bars add up exactly to the logit gap because the logits are linear in the concepts, which is the paper's faithfulness property:

$$
\operatorname{logit}_k(x) = \sum_{c=1}^{108} \underbrace{\sum_{h} \bar{a}_{hc}(x)\, \beta_{hck}}_{\text{contribution of concept } c} + b_k
$$

Here $\bar{a}_{hc}(x)$ is head $h$'s attention to concept $c$, averaged over the class token or the 196 patches, and $\beta_{hck}$ is concept $c$'s value vector in head $h$, projected onto class $k$. High attention is therefore not class support: a concept can draw attention while its $\beta$ favors the other class.

**Correct: Black-footed Albatross.** Brown primary color draws the most global attention, is annotated for this image, and gives the strongest push over the runner-up, Sooty Albatross. The same figure carries two warnings. Medium size draws the third-most attention yet favors Sooty Albatross, and buff leg color adds support while its attention lies mostly on the water.

In [ ]:
# One example in detail: the figure rendered from the checkpoint by scripts/visualize_test_explanations.py.
EXAMPLE_ID = 3  # Black-footed Albatross, a correct prediction; any image ID from the table in 5.2 works

explanations = json.loads((ARTIFACT_DIR / "examples.json").read_text())
by_id = {sample["image_id"]: sample for sample in explanations["examples"]}
assert EXAMPLE_ID in by_id, f"EXAMPLE_ID must be one of {sorted(by_id)}"
sample = by_id[EXAMPLE_ID]
if sample["correct"]:
    summary = (f"Correct: {sample['true_class']} at {sample['predicted_probability']:.1%}, "
               f"compared with the runner-up, {sample['comparison_class']}.")
else:
    summary = (f"Wrong: {sample['true_class']} predicted as {sample['predicted_class']} at "
               f"{sample['predicted_probability']:.1%}; the true species gets {sample['true_probability']:.2%}.")
show_image(ARTIFACT_DIR / sample["figure_png"], width=1100, caption_text=(
    f"<b>Figure 5.2</b> {summary} The signed bars add up to the logit margin of {sample['logit_margin']:.2f}."))

**Wrong: a Horned Grebe called an Eared Grebe, with high confidence.** Grey upperparts, attended across the water as well as the body, gives by far the largest push toward Eared Grebe; black upperparts favors the true species but is outweighed. None of the five most-attended global concepts is annotated for this image, and the top shape concept is perching-like, for a water bird. The explanation does not prevent the mistake, but it shows which concepts made it.

In [ ]:
# One example in detail: the figure rendered from the checkpoint by scripts/visualize_test_explanations.py.
EXAMPLE_ID = 2904  # Horned Grebe predicted as Eared Grebe, a confident error; any image ID from the table in 5.2 works

explanations = json.loads((ARTIFACT_DIR / "examples.json").read_text())
by_id = {sample["image_id"]: sample for sample in explanations["examples"]}
assert EXAMPLE_ID in by_id, f"EXAMPLE_ID must be one of {sorted(by_id)}"
sample = by_id[EXAMPLE_ID]
if sample["correct"]:
    summary = (f"Correct: {sample['true_class']} at {sample['predicted_probability']:.1%}, "
               f"compared with the runner-up, {sample['comparison_class']}.")
else:
    summary = (f"Wrong: {sample['true_class']} predicted as {sample['predicted_class']} at "
               f"{sample['predicted_probability']:.1%}; the true species gets {sample['true_probability']:.2%}.")
show_image(ARTIFACT_DIR / sample["figure_png"], width=1100, caption_text=(
    f"<b>Figure 5.3</b> {summary} The signed bars add up to the logit margin of {sample['logit_margin']:.2f}."))

<div style="border-top: 3px solid #ac543f; background: rgba(172, 84, 63, 0.07); padding: 12px 18px 14px 18px;"><div style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65; margin-bottom: 6px;">Key point</div><div style="font-family: Newsreader, Georgia, serif; font-size: 21px; line-height: 1.4;">The bars are exactly what the classifier computed, so the explanation is faithful by construction. Whether its concepts are right is a separate question: attention can sit on water, or on attributes the annotation does not list.</div></div>

### 5.4 Global concepts: where their attention depends on the image

Global concepts have no patch map in this model: only the class token attends to them, as the grey cells in Figure 1.2 show. To see which image regions a global concept's attention depends on, we compute a Grad-CAM map: the gradient of that concept's attention weight with respect to the patch tokens entering the last ViT block, the last place where the class token reads the patches. Each feature channel is weighted by its mean gradient over the 196 patches, and only the positive part is kept.

These maps are not the model's own explanation. They are a post-hoc approximation, the kind Section 1.2 criticizes: they show where the attention weight is sensitive to the image, not what the model computed, and each map is scaled to its own maximum. Read them next to the bar charts in 5.3, which remain the faithful account.

In [ ]:
# Figure 5.4: Grad-CAM maps for each example's three most-attended global concepts, from scripts/global_concept_gradcam.py.
GRADCAM_REPORT = ARTIFACT_DIR / "global_gradcam.json"
gradcam = json.loads(GRADCAM_REPORT.read_text()) if GRADCAM_REPORT.is_file() else None
if gradcam is not None and gradcam["mode"] == "checkpoint":
    assert gradcam["checkpoint_sha256"] == json.loads((ARTIFACT_DIR / "examples.json").read_text())["checkpoint_sha256"]
    show_image(ARTIFACT_DIR / "global_gradcam.png", width=900, caption_text=(
        "<b>Figure 5.4</b> For each of the six examples, its three most-attended global concepts. Color is Grad-CAM of "
        "the concept's attention weight with respect to the patch tokens entering the last ViT block, scaled to each "
        "map's own maximum. A post-hoc approximation, not attention the model computed."))
else:
    print("Not computed yet: these maps need the disjoint run's checkpoint, which this account cannot read.\n"
          "Once it is readable, run from the repository root:\n"
          "  .venv/bin/python scripts/global_concept_gradcam.py --checkpoint <path to binary_mnist_best_ckpt.ckpt>\n"
          "The script checks that the checkpoint matches the cached examples before writing the figure.")

### 5.5 Live: any example, redrawn from its saved tensors

The six examples' attention maps, contributions and logits are cached, so this cell redraws any of them without loading the model, and re-checks that the contributions rebuild the logits. Image 622 is a correct prediction whose strongest support, black leg color, is attended across the background.

<div style="border-top: 3px solid #ac543f; padding-top: 8px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; font-weight: 600;">Live demo</span><span style="font-size: 13.5px; opacity: 0.7;"> &middot; cached tensors, no model &middot; change <code>EXAMPLE_ID</code>, <code>SPATIAL_CONCEPT</code> or <code>TOP_K</code> and re-run</span></div>

In [ ]:
# Live: one of the six examples, redrawn from its cached tensors; no model or GPU needed.
EXAMPLE_ID = 622  # image ID: 3, 190, 622 (correct) or 434, 1593, 2904 (wrong)
SPATIAL_CONCEPT = None  # None = strongest peak attention; or part of a concept name, e.g. "wing color: black"
TOP_K = 8  # concepts with the largest signed contributions to draw, 1 to 12


def pretty(name):
    """'has_leg_color::black' -> 'leg color: black'."""
    return name.replace("has_", "").replace("_", " ").replace("::", ": ")


explanations = json.loads((ARTIFACT_DIR / "examples.json").read_text())
with np.load(ARTIFACT_DIR / "selected_examples.npz") as saved:
    tensors = dict(saved)
position = {sample["image_id"]: index for index, sample in enumerate(explanations["examples"])}
assert EXAMPLE_ID in position, f"EXAMPLE_ID must be one of {sorted(position)}"
assert 1 <= TOP_K <= 12, "TOP_K must be between 1 and 12"
i = position[EXAMPLE_ID]
sample = explanations["examples"][i]
global_names = [pretty(name) for name in explanations["global_attributes"]]
spatial_names = [pretty(name) for name in explanations["spatial_attributes"]]
if SPATIAL_CONCEPT is None:
    concept = sample["displayed_spatial_indices"][0]
else:
    matches = [j for j, name in enumerate(spatial_names) if pretty(SPATIAL_CONCEPT) in name]
    assert len(matches) == 1, f"{SPATIAL_CONCEPT!r} matches {len(matches)} spatial concepts; choose from {spatial_names}"
    concept = matches[0]

# The classifier is linear in the concepts, so contributions plus biases rebuild every class logit.
contributions = tensors["contributions"][i]  # 108 concepts (13 global, then 95 spatial) x 200 classes
logits = tensors["logits"][i]
np.testing.assert_allclose(contributions.sum(axis=0) + tensors["bias"], logits, atol=1e-4, rtol=1e-4)
pred, other = sample["predicted_label"], sample["comparison_label"]
margin = contributions[:, pred] - contributions[:, other]
top = np.argsort(-np.abs(margin), kind="stable")[:TOP_K]
names = ["G · " + name for name in global_names] + ["S · " + name for name in spatial_names]
values = [float(margin[j]) for j in top]
values += [float(margin.sum() - margin[top].sum()), float(tensors["bias"][pred] - tensors["bias"][other])]
labels = [names[j] for j in top] + ["All other concepts", "Classifier biases"]
probability = np.exp(logits - logits.max())
probability /= probability.sum()

fig, (ax_map, ax_global, ax_signed) = plt.subplots(
    1, 3, figsize=(15, 4.8), gridspec_kw={"width_ratios": [1, 1.15, 1.45]}, constrained_layout=True)
weights = tensors["spatial_attention"][i, :, concept].reshape(14, 14)  # one weight per 16 x 16-pixel patch
ax_map.imshow(tensors["images"][i])
ax_map.imshow(weights, cmap="magma", vmin=0, vmax=1, alpha=0.65, extent=(-0.5, 223.5, 223.5, -0.5),
              interpolation="nearest")
target_rows, target_cols = np.where(tensors["spatial_targets"][i, :, concept].reshape(14, 14) == 1)
ax_map.scatter((target_cols + 0.5) * 16 - 0.5, (target_rows + 0.5) * 16 - 0.5, s=35, facecolors="none",
               edgecolors="cyan", linewidths=1.2)
ax_map.set_title(f"S · {spatial_names[concept]}\npeak attention {weights.max():.2f}")
ax_map.axis("off")

attention = tensors["global_attention"][i]
order = np.argsort(-attention, kind="stable")[:5]
ax_global.barh(np.arange(5), attention[order], color=PALETTE["neg"])
for row, j in enumerate(order):
    ax_global.text(attention[j] + 0.01, row, f"{attention[j]:.2f}", va="center", fontsize=9)
ax_global.set_yticks(np.arange(5))
ax_global.set_yticklabels([global_names[j] + (" *" if tensors["global_targets"][i, j] == 1 else "") for j in order])
ax_global.invert_yaxis()
ax_global.set(xlim=(0, 1.05), xlabel="Share of the class token's attention", title="Top global concepts (* annotated)")
ax_global.grid(False, axis="y")

ax_signed.barh(np.arange(len(values)), values, color=[PALETTE["pos"] if v >= 0 else PALETTE["accent"] for v in values])
ax_signed.set_yticks(np.arange(len(values)))
ax_signed.set_yticklabels(labels)
ax_signed.invert_yaxis()
ax_signed.axvline(0, color="0.25", linewidth=0.8)
ax_signed.set(xlabel=f"Logit contribution, {sample['predicted_class']} minus {sample['comparison_class']}",
              title=f"The bars sum to the logit margin, {sum(values):.2f}")
ax_signed.grid(False, axis="y")
outcome = "correct" if sample["correct"] else f"wrong, true species {sample['true_class']}"
fig.suptitle(f"Image {EXAMPLE_ID} · predicted {sample['predicted_class']} at {probability[pred]:.1%} · {outcome}")
plt.show()
caption(f"<b>Figure 5.5</b> Redrawn from <code>selected_examples.npz</code>; the check above rebuilt all {logits.size} "
        f"logits from {contributions.shape[0]} concept contributions plus biases. Left: attention to "
        f"<i>{spatial_names[concept]}</i>, cyan rings on annotated patches. Right: green favors "
        f"{sample['predicted_class']}, terracotta favors {sample['comparison_class']}.")

<details><summary style="cursor: pointer; font-size: 15px;"><b>Backup:</b> how the six examples were chosen and computed</summary>

- **Chosen by rule, not by eye:** 64 evenly spaced positions in the disjoint test split, scanned in order. The first three correct and the first three wrong predictions with different true species were kept. Their probabilities describe these six images, not a new test accuracy.
- **Inference:** the disjoint run's selected checkpoint in evaluation mode, CPU float32, with the repository's test transform (resize to 224 × 224 and ImageNet normalization, no augmentation).
- **Contributions:** forward hooks capture each head's attention over the concepts, and the output layer projects each concept's value vector onto the 200 classes. Attention times projected value, summed over heads, gives one contribution per concept and class.
- **Code:** `scripts/visualize_test_explanations.py` extracts and renders everything in `explanation_artifacts/disjoint-45478721/`; its `--render-only` flag redraws the figures from the cached tensors. The cell below reruns the extraction.

</details>

In [ ]:
# Backup: recompute the six examples from the disjoint run's checkpoint. Slow on CPU; the cells above use the cache.
RERUN_EXPLANATIONS = False  # True loads the 3.7 GB checkpoint and overwrites ARTIFACT_DIR; then re-run 5.2 to 5.5

import subprocess

if RERUN_EXPLANATIONS:
    result = subprocess.run(
        [sys.executable, str(PROJECT_DIR / "scripts" / "visualize_test_explanations.py"), "--data-dir", str(DATA_DIR),
         "--output-dir", str(ARTIFACT_DIR), "--threads", str(CPU_THREADS)],
        capture_output=True, text=True)
    print(result.stdout[-3000:], result.stderr[-3000:], sep="\n")
    result.check_returncode()
else:
    print(f"Skipped: the figures above come from {ARTIFACT_DIR.relative_to(PROJECT_DIR)}. "
          "Set RERUN_EXPLANATIONS = True to rebuild them from the checkpoint.")